In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aabdollahii/corp-diabete-ont/final_ontology_corpus.txt


In [3]:
import os
import re
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
    )


PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Datasets: 5.0.0
Accelerate: 1.13.0
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [4]:
CORPUS_PATH = Path(
    "/kaggle/input/datasets/aabdollahii/corp-diabete-ont/"
    "final_ontology_corpus.txt"
)

OUTPUT_DIR = Path(
    "/kaggle/working/pubmedbert_diabetes_mlm"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Corpus file was not found: {CORPUS_PATH}"
    )

print("Corpus path:", CORPUS_PATH)
print("Output path:", OUTPUT_DIR)
print("Corpus size (MB):", round(CORPUS_PATH.stat().st_size / 1024**2, 2))


Corpus path: /kaggle/input/datasets/aabdollahii/corp-diabete-ont/final_ontology_corpus.txt
Output path: /kaggle/working/pubmedbert_diabetes_mlm
Corpus size (MB): 1.15


In [5]:
with open(CORPUS_PATH, "r", encoding="utf-8") as file:
    raw_sentences = [
        line.strip()
        for line in file
        if line.strip()
    ]

print("Raw sentences:", len(raw_sentences))

def normalize_sentence(sentence):
    sentence = re.sub(r"\s+", " ", sentence).strip()
    sentence = re.sub(
        r"\bPatient\s+\d+\b",
        "The patient",
        sentence,
        flags=re.IGNORECASE
    )
    sentence = re.sub(r"\s+([,.])", r"\1", sentence)
    return sentence

bad_patterns = [
    "synthetic concept",
    "synthetic individual",
    "clinical concept leads",
    "leads to clinical concept",
    "causes synthetic"
]

clean_sentences = []

for sentence in raw_sentences:
    sentence = normalize_sentence(sentence)
    sentence_lower = sentence.lower()
    word_count = len(sentence.split())
    
    if any(pattern in sentence_lower for pattern in bad_patterns):
        continue
    
    if word_count < 5:
        continue
    
    if word_count > 120:
        continue
    
    clean_sentences.append(sentence)

clean_sentences = list(dict.fromkeys(clean_sentences))

print("Clean unique sentences:", len(clean_sentences))
print(
    "Removed sentences:",
    len(raw_sentences) - len(clean_sentences)
)

if len(clean_sentences) < 100:
    raise ValueError(
        "The cleaned corpus has fewer than 100 sentences. "
        "Please inspect the corpus cleaning rules."
    )


Raw sentences: 20807
Clean unique sentences: 5818
Removed sentences: 14989


In [6]:
CLEAN_CORPUS_PATH = OUTPUT_DIR / "cleaned_diabetes_ontology_corpus.txt"

with open(CLEAN_CORPUS_PATH, "w", encoding="utf-8") as file:
    for sentence in clean_sentences:
        file.write(sentence + "\n")

print("Saved cleaned corpus:", CLEAN_CORPUS_PATH)


Saved cleaned corpus: /kaggle/working/pubmedbert_diabetes_mlm/cleaned_diabetes_ontology_corpus.txt


In [7]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": clean_sentences
})

dataset_split = dataset.train_test_split(
    test_size=0.10,
    seed=42,
    shuffle=True
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

print("Train sentences:", len(train_dataset))
print("Evaluation sentences:", len(eval_dataset))

display(train_dataset.to_pandas().head(10))


Train sentences: 5236
Evaluation sentences: 582


,text
0,The patient has a BMI of 32.7 and a glucose va...
1,The patient is 23 years old.
2,The patient has a BMI of 42.2.
3,The patient has a glucose value of 56 and a ne...
4,"The patient is 29 years old, has 0 pregnancies..."
5,The diabetes pedigree function for The patient...
6,The age of The patient is 21 years.
7,The patient has a BMI of 35.8 and a glucose va...
8,The patient is 27 years old and has a BMI of 2...
9,The patient has a BMI of 30.8 and a glucose va...


In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM
)

MODEL_NAME = (
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME
)

print("Tokenizer vocabulary size:", tokenizer.vocab_size)
print("Model loaded:", MODEL_NAME)
print("Model parameters:", f"{model.num_parameters():,}")


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from

Tokenizer vocabulary size: 30522
Model loaded: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Model parameters: 132,985,716


In [9]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_special_tokens_mask=True
    )

tokenized_train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training corpus"
)

tokenized_eval_dataset = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing evaluation corpus"
)

print(tokenized_train_dataset)
print(tokenized_eval_dataset)


Tokenizing training corpus:   0%|          | 0/5236 [00:00<?, ? examples/s]

Tokenizing evaluation corpus:   0%|          | 0/582 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 5236
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 582
})


In [10]:
from transformers import DataCollatorForLanguageModeling

MLM_PROBABILITY = 0.15
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_RATIO = 0.08
SEED = 42

if torch.cuda.is_available():
    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    )
else:
    gpu_memory_gb = 0

if gpu_memory_gb >= 14:
    PER_DEVICE_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 4
else:
    PER_DEVICE_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 8

USE_BF16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

USE_FP16 = (
    torch.cuda.is_available()
    and not USE_BF16
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY
)

effective_batch_size = (
    PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
)

print("Per-device batch size:", PER_DEVICE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", effective_batch_size)
print("FP16:", USE_FP16)
print("BF16:", USE_BF16)


Per-device batch size: 16
Gradient accumulation: 4
Effective batch size: 64
FP16: False
BF16: True


In [11]:
from transformers import (
    Trainer,
    TrainingArguments,
    set_seed
)

set_seed(SEED)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,

    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=USE_FP16,
    bf16=USE_BF16,

    dataloader_num_workers=2,
    report_to="none",

    seed=SEED,
    data_seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator,
)

print("Trainer initialized successfully.")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer initialized successfully.


In [12]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

training_result = trainer.train()

print(training_result)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,9.167773,1.069009
2,4.336611,1.063113
3,4.119122,0.988077


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=123, training_loss=5.322329389370553, metrics={'train_runtime': 190.6346, 'train_samples_per_second': 82.398, 'train_steps_per_second': 0.645, 'total_flos': 696502301517792.0, 'train_loss': 5.322329389370553, 'epoch': 3.0})


In [13]:
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print("Final model saved to:", FINAL_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /kaggle/working/pubmedbert_diabetes_mlm/final_model


In [14]:
from transformers import pipeline

fill_mask = pipeline(
    task="fill-mask",
    model=str(FINAL_MODEL_DIR),
    tokenizer=str(FINAL_MODEL_DIR),
    device=0 if torch.cuda.is_available() else -1
)

test_sentences = [
    "The patient has a [MASK] value of 168.",
    "The patient has a BMI of [MASK].",
    "A high glucose value may be associated with [MASK]."
]

for sentence in test_sentences:
    print("\nInput:", sentence)
    
    predictions = fill_mask(sentence)
    
    for prediction in predictions[:5]:
        print(
            f"Token: {prediction['token_str']!r} | "
            f"Score: {prediction['score']:.4f} | "
            f"Text: {prediction['sequence']}"
        )


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Input: The patient has a [MASK] value of 168.
Token: 'glucose' | Score: 0.9989 | Text: the patient has a glucose value of 168.
Token: 'bmi' | Score: 0.0008 | Text: the patient has a bmi value of 168.
Token: 'triglyceride' | Score: 0.0001 | Text: the patient has a triglyceride value of 168.
Token: 'insulin' | Score: 0.0001 | Text: the patient has a insulin value of 168.
Token: 'cholesterol' | Score: 0.0000 | Text: the patient has a cholesterol value of 168.

Input: The patient has a BMI of [MASK].
Token: '30' | Score: 0.0586 | Text: the patient has a bmi of 30.
Token: '32' | Score: 0.0579 | Text: the patient has a bmi of 32.
Token: '34' | Score: 0.0568 | Text: the patient has a bmi of 34.
Token: '33' | Score: 0.0535 | Text: the patient has a bmi of 33.
Token: '35' | Score: 0.0527 | Text: the patient has a bmi of 35.

Input: A high glucose value may be associated with [MASK].
Token: 'diabetes' | Score: 0.6389 | Text: a high glucose value may be associated with diabetes.
Token: 'obesity'

In [15]:
import torch

def predict_masked_sentences(sentences, model, tokenizer, top_k=3):
    model.eval()
    
    for sentence in sentences:
        inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
        
        mask_token_index = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
        
        mask_token_logits = logits[0, mask_token_index, :]
        top_tokens = torch.topk(mask_token_logits, top_k, dim=1).indices[0].tolist()
        
        print(f"Sentence: {sentence}")
        for i, token_id in enumerate(top_tokens):
            predicted_word = tokenizer.decode([token_id]).strip()
            print(f"  {i+1}: {predicted_word}")
        print("-" * 50)

# List of diabetes-focused cloze questions
cloze_questions = [
    "A fasting plasma glucose level of 126 mg/dL or higher is often diagnostic for [MASK].",
    "Patients with a Hemoglobin A1c level of 6.5% or above are typically classified as having [MASK].",
    "The primary metabolic characteristic of type 2 diabetes is [MASK] resistance.",
    "Obesity and a high body mass index (BMI) are significant [MASK] for developing type 2 diabetes.",
    "Regular physical activity is recommended to help improve [MASK] sensitivity in patients with diabetes.",
    "Gestational diabetes refers to high blood sugar that develops during [MASK].",
    "Prolonged hyperglycemia in diabetic patients can lead to damage in small blood vessels, a condition known as [MASK].",
    "Excessive thirst and frequent urination are classic clinical [MASK] of undiagnosed diabetes.",
    "Diabetic retinopathy is a complication of diabetes that specifically affects the [MASK]."
]

predict_masked_sentences(cloze_questions, model, tokenizer)


Sentence: A fasting plasma glucose level of 126 mg/dL or higher is often diagnostic for [MASK].
  1: diabetes
  2: t2dm
  3: t2d
--------------------------------------------------
Sentence: Patients with a Hemoglobin A1c level of 6.5% or above are typically classified as having [MASK].
  1: diabetes
  2: t2dm
  3: hyperglycemia
--------------------------------------------------
Sentence: The primary metabolic characteristic of type 2 diabetes is [MASK] resistance.
  1: insulin
  2: glucose
  3: leptin
--------------------------------------------------
Sentence: Obesity and a high body mass index (BMI) are significant [MASK] for developing type 2 diabetes.
  1: factors
  2: predictors
  3: risks
--------------------------------------------------
Sentence: Regular physical activity is recommended to help improve [MASK] sensitivity in patients with diabetes.
  1: insulin
  2: glucose
  3: glycemic
--------------------------------------------------
Sentence: Gestational diabetes refers to 